<a href="https://colab.research.google.com/github/lekha-8/E-commerce-/blob/main/Word_embedding_model_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import re
from collections import Counter
from sklearn.preprocessing import LabelEncoder
corpus =  """Word embeddings are a type of word representation that allows words
with similar meaning to have a similar representation. They are a distributed
representation for text that is perhaps one of the key breakthroughs for the
progress of deep learning in natural language processing."""
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return text.split()
words = preprocess(corpus)
window_size = 2
word_counts = Counter(words)
vocab = sorted(word_counts.keys())
vocab_size = len(vocab)
word_to_ix = {word: i for i, word in enumerate(vocab)}
ix_to_word = {i: word for word, i in word_to_ix.items()}
def generate_skipgram_pairs(words, window_size):
    pairs = []
    for i, word in enumerate(words):
        center_word = word
        for j in range(max(i - window_size, 0), min(i + window_size + 1, len(words))):
            if i != j:
                context_word = words[j]
                pairs.append((center_word, context_word))
    return pairs
pairs = generate_skipgram_pairs(words, window_size)
X = []
y = []
for center, context in pairs:
    X.append(word_to_ix[center])
    y.append(word_to_ix[context])
X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)
class Word2Vec(nn.Module):
    def __init__(self, vocab_size, embed_size):
        super(Word2Vec, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.output = nn.Linear(embed_size, vocab_size)
    def forward(self, input):
        embed = self.embedding(input)
        out = self.output(embed)
        return out
embed_size = 50
model = Word2Vec(vocab_size, embed_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 100
for epoch in range(epochs):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")
word_embeddings = model.embedding.weight.data

word = "word"
if word in word_to_ix:
    idx = word_to_ix[word]
    print(f"Embedding for '{word}':\n{word_embeddings[idx]}")
else:
    print(f"Word '{word}' not found in vocabulary.")


Epoch 20, Loss: 1.9574
Epoch 40, Loss: 1.7410
Epoch 60, Loss: 1.7230
Epoch 80, Loss: 1.7191
Epoch 100, Loss: 1.7175
Embedding for 'word':
tensor([ 0.4158,  0.2583,  1.0725,  0.2641, -0.0744, -2.1502, -1.4234,  0.1258,
        -0.9379, -1.5537,  0.2299, -1.5961, -0.6630, -0.1309,  1.6230, -0.8463,
        -1.4982,  0.4927,  1.2574,  1.3282,  2.2117, -0.4389,  0.1711, -1.5757,
        -0.6056,  0.2618, -1.4371,  0.3233, -0.3484,  2.1527, -1.3089,  0.4063,
         1.2705, -0.4239, -0.2622,  0.6738,  0.7143,  0.1963, -0.2723, -1.7504,
         0.8904,  0.7454,  0.1879, -0.6361,  0.5687, -0.2851, -0.5380,  1.4286,
        -1.4483,  1.1434])
